**Data Cleaning & Preprocessing**

INPUT FILES:
1) data/online_retail.csv
2) data/product_catalog.csv

OUTPUT FILES:
1) data/online_retail_cleaned.csv   — main clean dataset (19 columns)
2) data/cancellations.csv           — isolated return transactions
3) plots/quantity_distribution.png
4) plots/monthly_revenue_trend.png

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore') 

In [3]:
# ── file paths ─────────────────────────────────────────────────────────────
RAW_PATH     = 'data/online_retail.csv'
CATALOG_PATH = 'data/product_catalog.csv'
CLEAN_OUT    = 'data/online_retail_cleaned.csv'
CANCEL_OUT   = 'data/cancellations.csv'
PLOT_QTY     = 'plots/quantity_distribution.png'
PLOT_REV     = 'plots/monthly_revenue_trend.png'

**1. LOAD RAW DATA**

In [4]:
print("=" * 55)
print("  STEP 2 — DATA CLEANING & PREPROCESSING")
print("=" * 55)
 
print("\n[1/9] Loading raw data...")
df_raw = pd.read_csv(RAW_PATH)
 
print(f"      Shape       : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"      Memory      : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
 
print("\n      ── Data Types ──────────────────────")
print(df_raw.dtypes.to_string())
 
print("\n      ── Missing Values ───────────────────")
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
miss_df     = pd.DataFrame({'Count': missing, 'Pct %': missing_pct})
print(miss_df[miss_df['Count'] > 0].to_string())

  STEP 2 — DATA CLEANING & PREPROCESSING

[1/9] Loading raw data...
      Shape       : 91,648 rows × 10 columns
      Memory      : 36.5 MB

      ── Data Types ──────────────────────
InvoiceNo       object
StockCode        int64
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID      object
Country         object
Category        object
Revenue        float64

      ── Missing Values ───────────────────
             Count  Pct %
Description    916   1.00
CustomerID    7281   7.94


**2. REMOVE DUPLICATES**

In [5]:
print("\n[2/9] Checking for duplicate rows...")
 
n_dupes = df_raw.duplicated().sum()
df      = df_raw.drop_duplicates().copy()
 
print(f"      Exact duplicates found : {n_dupes:,}")
print(f"      Rows after dedup       : {len(df):,}")


[2/9] Checking for duplicate rows...
      Exact duplicates found : 0
      Rows after dedup       : 91,648


**3. SEPARATE CANCELLATIONS**

In [6]:
print("\n[3/9] Separating cancellations (C-prefix invoices)...")
# Cancellations have negative quantities and must not be mixed
# with sales — they would silently undercount revenue.
 
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
mask_cancel  = df['InvoiceNo'].str.startswith('C')
df_cancelled = df[mask_cancel].copy()
df_sales     = df[~mask_cancel].copy()
 
print(f"      Sales rows        : {len(df_sales):,}  ({len(df_sales)/len(df)*100:.1f}%)")
print(f"      Cancellation rows : {len(df_cancelled):,}  ({len(df_cancelled)/len(df)*100:.1f}%)")
 
print("\n      Cancellation rate by category (%):")
cancel_by_cat = (
    df_cancelled.groupby('Category')['InvoiceNo'].count()
    / df.groupby('Category')['InvoiceNo'].count() * 100
).round(2).sort_values(ascending=False)
print(cancel_by_cat.to_string())


[3/9] Separating cancellations (C-prefix invoices)...
      Sales rows        : 89,827  (98.0%)
      Cancellation rows : 1,821  (2.0%)

      Cancellation rate by category (%):
Category
Kitchen          2.13
Toys & Games     2.06
Gifts & Cards    2.05
Clothing         1.95
Garden           1.92
Stationery       1.92
Home Decor       1.88


**4. FIX DATA TYPES**

In [7]:
print("\n[4/9] Fixing data types...")
 
# InvoiceDate: string → datetime (critical for all time-based analysis)
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])
 
# StockCode: string identifier, not a numeric value
df_sales['StockCode'] = df_sales['StockCode'].astype(str)
 
print("      InvoiceDate : string → datetime")
print("      StockCode   : int    → string")


[4/9] Fixing data types...
      InvoiceDate : string → datetime
      StockCode   : int    → string


**5. HANDLE MISSING VALUES**

In [8]:
print("\n[5/9] Handling missing values...")
 
# 5a: Missing CustomerID → tag as 'Guest'
# These are valid guest-checkout transactions. We preserve their
# revenue data but tag them so they're excluded from customer-level
# analyses (RFM, cohort, churn).
before_guest = df_sales['CustomerID'].isna().sum()
df_sales['CustomerID'] = df_sales['CustomerID'].fillna('Guest')
print(f"      CustomerID  : {before_guest:,} nulls → filled with 'Guest'")
 
# 5b: Missing Description → fill from product catalog via StockCode
catalog = pd.read_csv(CATALOG_PATH)
catalog['StockCode'] = catalog['StockCode'].astype(str)
stock_desc_map = catalog.set_index('StockCode')['Description'].to_dict()
 
before_desc = df_sales['Description'].isna().sum()
df_sales['Description'] = df_sales.apply(
    lambda row: stock_desc_map.get(row['StockCode'], 'Unknown Product')
    if pd.isnull(row['Description']) else row['Description'],
    axis=1
)
print(f"      Description : {before_desc:,} nulls → filled from product catalog")
 
remaining_nulls = df_sales.isnull().sum().sum()
print(f"      Total missing values remaining: {remaining_nulls}")


[5/9] Handling missing values...
      CustomerID  : 7,081 nulls → filled with 'Guest'
      Description : 905 nulls → filled from product catalog
      Total missing values remaining: 0


**6. REMOVE INVALID QUANTITIES & PRICES**

In [9]:
print("\n[6/9] Removing invalid quantities and prices...")
# Non-cancelled rows should never have qty ≤ 0 or price ≤ 0.
# These are data entry errors, not valid transactions.
 
before        = len(df_sales)
invalid_qty   = (df_sales['Quantity']  <= 0).sum()
invalid_price = (df_sales['UnitPrice'] <= 0).sum()
 
df_sales = df_sales[(df_sales['Quantity'] > 0) & (df_sales['UnitPrice'] > 0)]
 
print(f"      Rows with Quantity  <= 0 : {invalid_qty:,}")
print(f"      Rows with UnitPrice <= 0 : {invalid_price:,}")
print(f"      Total removed            : {before - len(df_sales):,}")
print(f"      Remaining sales rows     : {len(df_sales):,}")


[6/9] Removing invalid quantities and prices...
      Rows with Quantity  <= 0 : 0
      Rows with UnitPrice <= 0 : 0
      Total removed            : 0
      Remaining sales rows     : 89,827


**7. OUTLIER TREATMENT — BULK B2B ORDERS**

In [10]:
print("\n[7/9] Flagging bulk B2B orders (outlier treatment)...")
# High-quantity orders (B2B bulk purchases) are VALID transactions.
# We FLAG them rather than delete so they can be segmented separately
# without distorting retail customer analysis.
# Method: IQR-based threshold (Q3 + 3×IQR)
 
Q1  = df_sales['Quantity'].quantile(0.25)
Q3  = df_sales['Quantity'].quantile(0.75)
IQR = Q3 - Q1
bulk_threshold = Q3 + 3 * IQR
 
df_sales['IsBulkOrder'] = df_sales['Quantity'] > bulk_threshold
bulk_count = df_sales['IsBulkOrder'].sum()
 
print(f"      Q1 = {Q1:.0f}  |  Q3 = {Q3:.0f}  |  IQR = {IQR:.1f}")
print(f"      Bulk threshold (Q3 + 3×IQR) : {bulk_threshold:.0f} units")
print(f"      Bulk orders flagged          : {bulk_count:,} ({bulk_count/len(df_sales)*100:.1f}%)")
 
# Plot: Quantity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_sales['Quantity'], bins=60,
             color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Quantity Distribution (Full range)')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Frequency')
for sp in ['top', 'right']: axes[0].spines[sp].set_visible(False)
 
axes[1].hist(df_sales['Quantity'].clip(upper=100), bins=60,
             color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].set_title('Quantity Distribution (Clipped at 100 for visibility)')
axes[1].set_xlabel('Quantity')
for sp in ['top', 'right']: axes[1].spines[sp].set_visible(False)
 
plt.suptitle('Quantity Distribution Before & After Clipping', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_QTY, dpi=150, bbox_inches='tight')
plt.close()
print(f"      Plot saved → {PLOT_QTY}")


[7/9] Flagging bulk B2B orders (outlier treatment)...
      Q1 = 7  |  Q3 = 19  |  IQR = 12.0
      Bulk threshold (Q3 + 3×IQR) : 55 units
      Bulk orders flagged          : 3,067 (3.4%)
      Plot saved → plots/quantity_distribution.png


**8. FEATURE ENGINEERING**

In [11]:
print("\n[8/9] Feature engineering — extracting time features...")
 
df_sales['Year']      = df_sales['InvoiceDate'].dt.year
df_sales['Month']     = df_sales['InvoiceDate'].dt.month
df_sales['MonthName'] = df_sales['InvoiceDate'].dt.strftime('%b')
df_sales['Day']       = df_sales['InvoiceDate'].dt.day
df_sales['DayOfWeek'] = df_sales['InvoiceDate'].dt.day_name()
df_sales['IsWeekend'] = df_sales['InvoiceDate'].dt.dayofweek >= 5
df_sales['Quarter']   = df_sales['InvoiceDate'].dt.quarter
df_sales['YearMonth'] = df_sales['InvoiceDate'].dt.to_period('M').astype(str)
 
# Recalculate Revenue on fully clean data
df_sales['Revenue'] = df_sales['Quantity'] * df_sales['UnitPrice']
 
print("      Features added : Year, Month, MonthName, Day, DayOfWeek,")
print("                       IsWeekend, Quarter, YearMonth, IsBulkOrder, Revenue")
print(f"      Final columns  : {df_sales.shape[1]}")
 
# Plot: Monthly revenue trend (sanity check for seasonality)
monthly_rev = (
    df_sales.groupby('YearMonth')['Revenue']
    .sum()
    .reset_index()
    .sort_values('YearMonth')
)
 
fig, ax = plt.subplots(figsize=(14, 4))
colors = [
    '#d64e4e' if ('Nov' in ym or 'Dec' in ym) else 'steelblue'
    for ym in monthly_rev['YearMonth']
]
ax.bar(monthly_rev['YearMonth'], monthly_rev['Revenue'] / 1e3,
       color=colors, edgecolor='white', linewidth=0.4)
ax.set_title('Monthly Revenue (£ thousands) — Red = Nov/Dec seasonal peaks',
             fontsize=12)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£K)')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}K')
)
plt.xticks(rotation=45, ha='right', fontsize=8)
for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig(PLOT_REV, dpi=150, bbox_inches='tight')
plt.close()
print(f"      Plot saved → {PLOT_REV}")


[8/9] Feature engineering — extracting time features...
      Features added : Year, Month, MonthName, Day, DayOfWeek,
                       IsWeekend, Quarter, YearMonth, IsBulkOrder, Revenue
      Final columns  : 19
      Plot saved → plots/monthly_revenue_trend.png


**9. SAVE OUTPUTS**

In [12]:
print("\n[9/9] Saving cleaned datasets...")
 
df_sales.to_csv(CLEAN_OUT, index=False)
df_cancelled.to_csv(CANCEL_OUT, index=False)
 
print(f"      Saved : {CLEAN_OUT}")
print(f"      Saved : {CANCEL_OUT}")


[9/9] Saving cleaned datasets...
      Saved : data/online_retail_cleaned.csv
      Saved : data/cancellations.csv


**FINAL SUMMARY**

In [13]:
print()
print("=" * 55)
print("  CLEANING SUMMARY")
print("=" * 55)
print(f"  Raw rows              : {len(df_raw):>10,}")
print(f"  Duplicates removed    : {n_dupes:>10,}")
print(f"  Cancellations removed : {len(df_cancelled):>10,}  → cancellations.csv")
print(f"  Final clean rows      : {len(df_sales):>10,}")
print(f"  Final columns         : {df_sales.shape[1]:>10,}")
print()
print(f"  Unique Customers      : {df_sales['CustomerID'].nunique():>10,}")
print(f"    of which Guests     : {(df_sales['CustomerID']=='Guest').sum():>10,}")
print(f"  Unique Products       : {df_sales['StockCode'].nunique():>10,}")
print(f"  Date Range            : {df_sales['InvoiceDate'].min().date()} "
      f"→ {df_sales['InvoiceDate'].max().date()}")
print(f"  Total Revenue         : £{df_sales['Revenue'].sum():>12,.2f}")
print(f"  Avg Order Value       : "
      f"£{df_sales.groupby('InvoiceNo')['Revenue'].sum().mean():>10,.2f}")
print(f"  Bulk orders flagged   : {df_sales['IsBulkOrder'].sum():>10,}")
print()
 
nulls = df_sales.isnull().sum()
if nulls.sum() == 0:
    print("  Missing values  : zero")
else:
    print("  Missing values  :")
    print(nulls[nulls > 0].to_string())
 
print()
print("""
  CLEANING DECISIONS REFERENCE
  ──────────────────────────────────────────────────────
  Issue                  Action            Rationale
  ──────────────────────────────────────────────────────
  Duplicate rows         Dropped           Prevent double-counting
  Cancellations          Isolated to file  Keep for return analysis
  Missing CustomerID     Tagged 'Guest'    Preserve revenue data
  Missing Description    Catalog lookup    Maintain product context
  Qty / Price <= 0       Dropped           Data entry errors
  Bulk orders            Flagged boolean   Valid B2B, not errors
  InvoiceDate (string)   -> datetime       Enable time analysis
  New features           +9 columns        Power EDA & ML
  ──────────────────────────────────────────────────────
""")


  CLEANING SUMMARY
  Raw rows              :     91,648
  Duplicates removed    :          0
  Cancellations removed :      1,821  → cancellations.csv
  Final clean rows      :     89,827
  Final columns         :         19

  Unique Customers      :      4,696
    of which Guests     :      7,081
  Unique Products       :         93
  Date Range            : 2022-12-01 → 2024-11-30
  Total Revenue         : £20,035,718.48
  Avg Order Value       : £    731.20
  Bulk orders flagged   :      3,067

  Missing values  : zero


  CLEANING DECISIONS REFERENCE
  ──────────────────────────────────────────────────────
  Issue                  Action            Rationale
  ──────────────────────────────────────────────────────
  Duplicate rows         Dropped           Prevent double-counting
  Cancellations          Isolated to file  Keep for return analysis
  Missing CustomerID     Tagged 'Guest'    Preserve revenue data
  Missing Description    Catalog lookup    Maintain product context
  